# 🦈 Análise Exploratória de Ataques de Tubarões

Este notebook realiza uma análise exploratória completa dos dados do Global Shark Attack File.

## Objetivos
- Explorar padrões temporais e geográficos
- Identificar espécies mais envolvidas
- Analisar atividades de risco
- Visualizar tendências e estatísticas

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Carregamento e Limpeza dos Dados

In [ ]:
# Carregar dados brutos
df = pd.read_csv('../data/attacks.csv', encoding='latin-1', low_memory=False)

print(f"Dimensões dos dados: {df.shape}")
print(f"\nColunas disponíveis: {df.columns.tolist()}")

In [ ]:
# Visualizar primeiras linhas
df.head()

In [ ]:
# Informações sobre os dados
df.info()

In [ ]:
# Executar limpeza de dados
import sys
sys.path.append('../src')
from data_cleaning import clean_data

df_clean = clean_data()

## 2. Estatísticas Descritivas

In [ ]:
# Estatísticas básicas
df_clean.describe()

In [ ]:
# Valores faltantes
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean)) * 100

missing_df = pd.DataFrame({
    'Valores Faltantes': missing,
    'Percentual': missing_pct
}).sort_values('Valores Faltantes', ascending=False)

missing_df[missing_df['Valores Faltantes'] > 0].head(10)

## 3. Análise Temporal

In [ ]:
# Evolução ao longo do tempo
if 'year' in df_clean.columns:
    attacks_by_year = df_clean[df_clean['year'] >= 1900].groupby('year').size()
    
    plt.figure(figsize=(14, 6))
    plt.plot(attacks_by_year.index, attacks_by_year.values, linewidth=2)
    plt.fill_between(attacks_by_year.index, attacks_by_year.values, alpha=0.3)
    plt.title('Evolução dos Ataques de Tubarões (1900-Presente)', fontsize=16, fontweight='bold')
    plt.xlabel('Ano')
    plt.ylabel('Número de Ataques')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Distribuição por década
if 'year' in df_clean.columns:
    df_clean['decade'] = (df_clean['year'] // 10) * 10
    attacks_by_decade = df_clean[df_clean['decade'] >= 1900].groupby('decade').size()
    
    plt.figure(figsize=(14, 6))
    attacks_by_decade.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Ataques por Década', fontsize=16, fontweight='bold')
    plt.xlabel('Década')
    plt.ylabel('Número de Ataques')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.show()

In [ ]:
# Sazonalidade (por mês)
if 'month' in df_clean.columns:
    month_names = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
                   'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    
    attacks_by_month = df_clean['month'].value_counts().sort_index()
    
    plt.figure(figsize=(12, 6))
    plt.bar(range(1, 13), [attacks_by_month.get(i, 0) for i in range(1, 13)],
            color='coral', edgecolor='black')
    plt.xticks(range(1, 13), month_names)
    plt.title('Distribuição de Ataques por Mês', fontsize=16, fontweight='bold')
    plt.xlabel('Mês')
    plt.ylabel('Número de Ataques')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

## 4. Análise Geográfica

In [ ]:
# Top países com mais ataques
if 'country' in df_clean.columns:
    top_countries = df_clean['country'].value_counts().head(15)
    
    plt.figure(figsize=(12, 8))
    top_countries.plot(kind='barh', color='crimson')
    plt.title('Top 15 Países com Mais Ataques', fontsize=16, fontweight='bold')
    plt.xlabel('Número de Ataques')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.show()
    
    print("\nTop 10 países:")
    print(top_countries.head(10))

## 5. Análise por Espécie

In [ ]:
# Espécies mais envolvidas
species_col = 'species' if 'species' in df_clean.columns else 'species_'

if species_col in df_clean.columns:
    species_data = df_clean[df_clean[species_col].notna() & (df_clean[species_col] != '')]
    top_species = species_data[species_col].value_counts().head(10)
    
    plt.figure(figsize=(12, 8))
    top_species.plot(kind='barh', color=sns.color_palette('husl', 10))
    plt.title('Top 10 Espécies de Tubarões Envolvidas', fontsize=16, fontweight='bold')
    plt.xlabel('Número de Ataques')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.show()
    
    print("\nTop 10 espécies:")
    print(top_species)

## 6. Análise por Atividade

In [ ]:
# Ataques por tipo de atividade
if 'activity_category' in df_clean.columns:
    activity_counts = df_clean['activity_category'].value_counts()
    
    plt.figure(figsize=(10, 10))
    plt.pie(activity_counts.values, labels=activity_counts.index,
            autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set3'))
    plt.title('Distribuição por Tipo de Atividade', fontsize=16, fontweight='bold')
    plt.show()
    
    print("\nDistribuição por atividade:")
    print(activity_counts)

## 7. Análise de Fatalidade

In [ ]:
# Taxa de fatalidade geral
if 'is_fatal' in df_clean.columns:
    fatal_data = df_clean[df_clean['is_fatal'].notna()]
    fatal_counts = fatal_data['is_fatal'].value_counts()
    
    total_fatal = fatal_counts.get(1, 0)
    total_non_fatal = fatal_counts.get(0, 0)
    fatality_rate = (total_fatal / (total_fatal + total_non_fatal)) * 100
    
    print(f"Taxa de fatalidade: {fatality_rate:.2f}%")
    print(f"Ataques fatais: {total_fatal:,}")
    print(f"Ataques não-fatais: {total_non_fatal:,}")
    
    plt.figure(figsize=(8, 8))
    plt.pie([total_non_fatal, total_fatal], 
            labels=['Não Fatal', 'Fatal'],
            autopct='%1.1f%%',
            colors=['green', 'red'],
            explode=(0.05, 0.05),
            startangle=90)
    plt.title('Taxa de Fatalidade', fontsize=16, fontweight='bold')
    plt.show()

In [ ]:
# Fatalidade por atividade
if 'is_fatal' in df_clean.columns and 'activity_category' in df_clean.columns:
    fatality_by_activity = df_clean.groupby('activity_category')['is_fatal'].agg(['sum', 'count', 'mean'])
    fatality_by_activity.columns = ['Fatal', 'Total', 'Taxa']
    fatality_by_activity['Taxa'] = fatality_by_activity['Taxa'] * 100
    fatality_by_activity = fatality_by_activity.sort_values('Taxa', ascending=False)
    
    print("\nTaxa de fatalidade por atividade:")
    print(fatality_by_activity)

## 8. Correlações e Insights

In [ ]:
# Heatmap de correlação (colunas numéricas)
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 1:
    plt.figure(figsize=(10, 8))
    correlation_matrix = df_clean[numeric_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Matriz de Correlação', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Conclusões

Com base nas análises realizadas, podemos observar:

1. **Tendência Temporal**: [Adicione suas observações]
2. **Geografia**: [Adicione suas observações]
3. **Espécies**: [Adicione suas observações]
4. **Atividades de Risco**: [Adicione suas observações]
5. **Fatalidade**: [Adicione suas observações]